# Extended Data Fig. 4

ONT, PacBio and whole-genome bisulfite sequencing across the genome.

Panels d to g are drawn in R, and the R source is in the cells. Panels a to c are the one
place in this figure where the paper's script cannot run here: it reads three 450 MB point
tables.

In [ ]:
import sys; sys.path.insert(0, "../scripts"); sys.path.insert(0, "../scripts/paper")
from pathlib import Path
from nbkit import setup, paper_py, paper_r, run_r_source, harvest, show, have_r_package, DATA

OUT = setup("../figures/notebook")
print("panels will be written to", OUT)

# %%R cells below run R natively in this kernel through rpy2, so the R panels are R code
# in the notebook rather than a string that gets written to a file.
%load_ext rpy2.ipython

## a-c - per-site methylation, every pair of platforms

**Not the paper's drawing code.** The hexagons are the ones the paper's own `hexbin()` computed, exported once from 56 to 62 million sites. Everything else in this notebook is the paper's.

In [ ]:
import panels as P
import matplotlib.pyplot as plt
P.style()

specs = [("ed4_a_ont_vs_wgbs", "hexbin_a_ont_vs_wgbs_nm_stats.txt", "WGBS methylation (%)", "ONT methylation (%)"),
         ("ed4_b_pacbio_vs_wgbs", "hexbin_b_pacbio_vs_wgbs_nm_stats.txt", "WGBS methylation (%)", "PacBio methylation (%)"),
         ("ed4_c_ont_vs_pacbio", "hexbin_c_ont_vs_pacbio_nm_stats.txt", "PacBio methylation (%)", "ONT methylation (%)")]

fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.3))
for (key, statfile, xl, yl), ax in zip(specs, axes):
    P.hexbin(key, xl, yl, P.read_hexbin_stats(DATA / "ed_fig4" / statfile), ax=ax)
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(OUT / f"ed4abc_hexbins.{ext}", bbox_inches="tight")
plt.close(fig)
show("ed4abc_hexbins.png")

## d - haplotype DMRs along the autosomes, ONT

Drawn at the published cutoff: |HP1 - HP2| at least 50 percentage points, q at most 0.01. The input holds the DMRs that already clear it, 45,339 of 2.6 million tiles; the script filters again at the same threshold, so what it draws is unchanged.

In [ ]:
%%R
## scripts/paper/ideogram_dmr_blue.R, run as R. Edit any line and re-run the cell.
Sys.setenv(IDEOGRAM_METHDIFF = "50")
Sys.setenv(IDEOGRAM_QVALUE = "0.01")
Sys.setenv(IDEOGRAM_BASE_PT = "7")
Sys.setenv(IDEOGRAM_TITLE_PT = "7")
Sys.setenv(IDEOGRAM_WIDTH_IN = "3.5")
Sys.setenv(IDEOGRAM_HEIGHT_IN = "3.3")
Sys.setenv(IDEOGRAM_LEGEND_KEY = "2")
Sys.setenv(IDEOGRAM_TITLE = "")
Sys.setenv(IDEOGRAM_WDIR = file.path(Sys.getenv("LV_DATA"), "ed_fig4/dmr/ont"))

#!/usr/bin/env Rscript
rm(list = ls())
require(GenomicRanges)
require(ggbio)

# Visualize Hyper/Hypo DMR for HG002 ONT on T2T genome with gap region annotations

## What the ideogram draws. These two numbers are the panel: a DMR is plotted only if it
## clears both. They are NOT the function defaults further down (25 and 0.01) - line ~196
## passes these in and overrides those, which is how the panels came to be drawn at 10 and
## 0.05 while every count quoted beside them was computed at 25 and 0.01. Anything that
## reports a DMR count must read these same variables, which is why they are overridable:
## the number in the log and the dots on the page then cannot drift apart.
##
## 2026-10-03: raised to 50 and 0.01 on the user's call. At 10 percent the ideogram is a
## solid band on every chromosome and says nothing; 50 percent leaves the haplotype
## differences that are actually worth looking at.
methDiffCutoff <- as.numeric(Sys.getenv("IDEOGRAM_METHDIFF", "50"))
methDiffQValue <- as.numeric(Sys.getenv("IDEOGRAM_QVALUE",   "0.01"))
## Typography and canvas. The defaults reproduce the 12 pt, 8 x 7 in panel exactly; the
## Nature Methods render (paper_figures/fig_ed4) sets IDEOGRAM_BASE_PT=7 and a 3.5 in
## canvas, and blanks the title because the deck carries the panel caption.
basePt   <- as.numeric(Sys.getenv("IDEOGRAM_BASE_PT", "12"))
titlePt  <- as.numeric(Sys.getenv("IDEOGRAM_TITLE_PT", as.character(basePt + 2)))
figW     <- as.numeric(Sys.getenv("IDEOGRAM_WIDTH_IN", "8"))
figH     <- as.numeric(Sys.getenv("IDEOGRAM_HEIGHT_IN", "7"))
legendKey <- as.numeric(Sys.getenv("IDEOGRAM_LEGEND_KEY", "4"))
titleText <- Sys.getenv("IDEOGRAM_TITLE", "__default__")
cat(sprintf("[CUTOFF] methdiff >= %g, qvalue <= %g\n", methDiffCutoff, methDiffQValue))

## Parameterised copy of hpc_test/analysis/Figure4/IdeomgramPlot_methykit_{ont,pacbio}_hg002.R.
## The two originals differ only in platform name, input path, title and output name, so they
## are one script here. The originals are left untouched.
##
## Usage: 02_ideogram_dmr_blue.R <ont|pacbio> <outdir>
## commandArgs replaced for the notebook; everything else is the paper's file
argv <- c("ont", Sys.getenv("LV_OUT"))
platform <- if (length(argv) >= 1) argv[1] else "ont"
outdir   <- if (length(argv) >= 2) argv[2] else "."
stopifnot(platform %in% c("ont", "pacbio"))
dir.create(outdir, showWarnings = FALSE, recursive = TRUE)
PLATFORM_LABEL <- if (platform == "ont") "ONT" else "PacBio"

## The default is the June 2026 run. Its PacBio branch is primrose, so the PacBio
## ideogram must be pointed at a jasmine DMR directory instead; the ONT branch there
## is unaffected and stays the default. IDEOGRAM_WDIR overrides, nothing else changes.
wdir <- Sys.getenv('IDEOGRAM_WDIR',
                   '/project2/sli68423_1316/projects/long_verse/results/2026_06_16_phasing_dmr')
dir.create(wdir, showWarnings = FALSE, recursive = TRUE)
setwd(wdir)

chm13_chrom_size_file <- '/project2/sli68423_1316/users/yang/reference/chm13v2.0/chm13v2.0.chrom.sizes'
chm13_df <- read.table(chm13_chrom_size_file, sep = "\t", header = FALSE)

chm13.chr.len <- chm13_df[[2]]
names(chm13.chr.len) <- chm13_df[[1]]

# keep autosomes chr1-chr22 only
autosomes <- paste0("chr", 1:22)
chm13.chr.len <- chm13.chr.len[autosomes[autosomes %in% names(chm13.chr.len)]]

# load the methylKit DMR table. The originals name a plain .tsv; the files on disk were
# gzipped after that run, so take whichever is present rather than failing on the name.
dmr_file <- file.path(wdir, sprintf('methylkit_call/%s/hg002_%s_HP1_vs_HP2_methylkit_DMR_all.tsv',
                                    platform, platform))
if (!file.exists(dmr_file) && file.exists(paste0(dmr_file, '.gz'))) {
    dmr_file <- paste0(dmr_file, '.gz')
}
stopifnot(file.exists(dmr_file))
cat("DMR table:", dmr_file, "\n")
dmr_df <- read.table(gzfile(dmr_file), sep = "\t", header = TRUE,
                     stringsAsFactors = FALSE, check.names = FALSE)
dmr_df <- dmr_df[dmr_df$chr %in% names(chm13.chr.len),]

# gap regions (TSV with header: chrom, chromStart, chromEnd)
t2t_gap_bed_file <- file.path(wdir, 'AH107353_T2T_hgUnique.tsv')
t2t_gap_bed_df <- read.table(t2t_gap_bed_file, sep = "\t", header = TRUE, stringsAsFactors = FALSE)
t2t_gap_bed_df <- t2t_gap_bed_df[t2t_gap_bed_df$chrom %in% names(chm13.chr.len),]
t2t_gap_granges <- GRanges(
    seqnames = t2t_gap_bed_df$chrom,
    ranges = IRanges(start = t2t_gap_bed_df$chromStart, end = t2t_gap_bed_df$chromEnd)
)

filter_dmr <- function(dmr_df, difference, qvalue, type) {
    sig <- abs(dmr_df$meth.diff) >= difference & dmr_df$qvalue <= qvalue
    if (type == "hyper") {
        dmr_df[sig & dmr_df$meth.diff > 0,]
    } else {
        dmr_df[sig & dmr_df$meth.diff < 0,]
    }
}

dmr_to_granges <- function(dmr_df) {
    GRanges(
        seqnames = dmr_df$chr,
        ranges = IRanges(start = dmr_df$start, end = dmr_df$end),
        meth.diff = dmr_df$meth.diff
    )
}

gap_ylim <- c(-100, 100)

# colorblind-friendly palette (Wong): pink vs BLUE; gap stays red.
#
# The original used Wong yellow #F0E442 for hypo. Wong is colourblind safe, but yellow on
# white has almost no luminance contrast, and on a printed poster at viewing distance the
# hypo points disappear. Wong blue #0072B2 keeps the palette colourblind safe and is the
# darkest colour in it, so the two DMR classes separate by lightness as well as by hue,
# which is what survives printing and photocopying.
hyper_col <- "#CC79A7"  # pink
hypo_col <- "#0072B2"   # blue, was #F0E442 yellow
gap_col <- "red"

ideoDMC <- function(dmr_df, chrom.length, difference = 25,
                    qvalue = 0.01, circos = FALSE, title = "Test title",
                    hyper.col = hyper_col,
                    hypo.col = hypo_col,
                    gap.col = gap_col) {
    myIdeo <- GRanges(seqnames = names(chrom.length),
                      ranges = IRanges(start = 1, width = chrom.length))
    seqlevels(myIdeo) <- names(chrom.length)
    seqlengths(myIdeo) <- chrom.length

    hyper <- filter_dmr(dmr_df, difference, qvalue, "hyper")
    hypo <- filter_dmr(dmr_df, difference, qvalue, "hypo")

    g.per <- dmr_to_granges(hyper)
    seqlevels(g.per) <- seqlevels(myIdeo)
    seqlengths(g.per) <- chrom.length

    g.po <- dmr_to_granges(hypo)
    seqlevels(g.po) <- seqlevels(myIdeo)
    seqlengths(g.po) <- chrom.length

    values(g.po)$id <- "Hypo"
    values(g.per)$id <- "Hyper"
    legend_levels <- c("Hyper", "Hypo", "Gap region")
    values(g.po)$id <- factor(values(g.po)$id, levels = legend_levels)
    values(g.per)$id <- factor(values(g.per)$id, levels = legend_levels)

    # invisible dummy point so "Gap region" appears in the color legend
    g.gap.legend <- GRanges(
        seqnames = names(chrom.length)[1],
        ranges = IRanges(start = 1, width = 1),
        meth.diff = gap_ylim[1] - 1,
        id = factor("Gap region", levels = legend_levels)
    )
    seqlevels(g.gap.legend) <- seqlevels(myIdeo)
    seqlengths(g.gap.legend) <- chrom.length
    g.dmr <- c(g.po, g.per, g.gap.legend)

    if (circos) {
        p <- ggplot() + layout_circle(myIdeo, geom = "ideo", fill = "gray70",
                                      radius = 39, trackWidth = 2)

        p <- p +
            layout_circle(c(g.po, g.per), geom = "point",
                          size = 1, aes(x = midpoint,
                                        y = meth.diff, color = id), radius = 25, trackWidth = 30) +
            scale_colour_manual(values = c("Hyper" = hyper.col, "Hypo" = hypo.col))
        p3 <- p +
            layout_circle(myIdeo, geom = "text", aes(label = seqnames),
                          vjust = 0, radius = 55, trackWidth = 7) +
            labs(title = title)

    } else {
        legend_shapes <- c("Hyper" = 16, "Hypo" = 16, "Gap region" = 22)
        legend_title <- "Annotation"
        p1 <- ggplot() + layout_karyogram(myIdeo)
        p2 <- p1 +
            layout_karyogram(g.dmr, geom = "point", size = 0.01,
                             aes(x = midpoint,
                                 y = meth.diff, color = id)) +
            scale_colour_manual(legend_title,
                                breaks = legend_levels,
                                values = c("Hyper" = hyper.col,
                                           "Hypo" = hypo.col,
                                           "Gap region" = gap.col)) +
            labs(title = title) +
            xlab("Genomic coordinates") +
            ylab("Methylation difference (%)") +
            ## ggplot's base_size is not a floor: axis.text defaults to rel(0.8), so a
            ## base of 11 puts the chromosome strip labels and both axes at 8.8 pt. Only
            ## plot.title was ever set here, which is why the panel sat at 9 pt while
            ## reading as finished. Every text element is named explicitly instead.
            theme(
                plot.title   = if (nzchar(title)) element_text(hjust = 0.5, size = titlePt) else element_blank(),
                axis.title   = element_text(size = basePt),
                axis.text    = element_text(size = basePt),
                strip.text   = element_text(size = basePt),
                legend.title = element_text(size = basePt),
                legend.text  = element_text(size = basePt),
                legend.key.size = unit(basePt * 1.2, "pt")
            ) +
            guides(color = guide_legend(
                title = legend_title,
                override.aes = list(
                    shape = unname(legend_shapes[legend_levels]),
                    fill = c(hyper.col, hypo.col, NA),
                    colour = c(hyper.col, hypo.col, gap.col),
                    alpha = c(1, 1, 1),
                    size = c(legendKey, legendKey, legendKey)
                )
            ))
        p3 <- p2 +
            layout_karyogram(t2t_gap_granges,
                             geom = "rect",
                             rect.height = abs(gap_ylim[2] - gap_ylim[1]) / 2,
                             ylim = gap_ylim,
                             color = gap.col, fill = NA) +
            coord_cartesian(ylim = gap_ylim)
    }

    return(p3)
}

ideoPlot <- ideoDMC(dmr_df, chrom.length = chm13.chr.len,
                    difference = methDiffCutoff, qvalue = methDiffQValue,
                    circos = FALSE,
                    title = if (identical(titleText, "__default__"))
                        sprintf("%s: DMRs of Haplotypes in HG002", PLATFORM_LABEL) else titleText)

ideoPlot

stem <- file.path(outdir, sprintf('Ideogram_dmr_t2t_hg002_%s_blue', platform))

## cairo_pdf, not pdf(): the base device does not embed fonts, so an Arial panel opens in
## whatever the reader substitutes. Same reason png() is given the cairo type. The size is
## 8 x 7 rather than 8 x 6: at 12 pt the 24 chromosome strip labels need the extra inch.
cairo_pdf(paste0(stem, '.pdf'), width = figW, height = figH)
print(ideoPlot)
dev.off()

png(paste0(stem, '.png'), width = figW, height = figH, units = "in", res = 400, type = "cairo")
print(ideoPlot)
dev.off()

cat("wrote", paste0(stem, c('.pdf', '.png')), "\n")
cat("hyper", hyper_col, " hypo", hypo_col, "\n")

## e - the same for PacBio

The same script, the other platform's DMR table.

In [ ]:
%%R
## scripts/paper/ideogram_dmr_blue.R, run as R. Edit any line and re-run the cell.
Sys.setenv(IDEOGRAM_METHDIFF = "50")
Sys.setenv(IDEOGRAM_QVALUE = "0.01")
Sys.setenv(IDEOGRAM_BASE_PT = "7")
Sys.setenv(IDEOGRAM_TITLE_PT = "7")
Sys.setenv(IDEOGRAM_WIDTH_IN = "3.5")
Sys.setenv(IDEOGRAM_HEIGHT_IN = "3.3")
Sys.setenv(IDEOGRAM_LEGEND_KEY = "2")
Sys.setenv(IDEOGRAM_TITLE = "")
Sys.setenv(IDEOGRAM_WDIR = file.path(Sys.getenv("LV_DATA"), "ed_fig4/dmr/pacbio"))

#!/usr/bin/env Rscript
rm(list = ls())
require(GenomicRanges)
require(ggbio)

# Visualize Hyper/Hypo DMR for HG002 ONT on T2T genome with gap region annotations

## What the ideogram draws. These two numbers are the panel: a DMR is plotted only if it
## clears both. They are NOT the function defaults further down (25 and 0.01) - line ~196
## passes these in and overrides those, which is how the panels came to be drawn at 10 and
## 0.05 while every count quoted beside them was computed at 25 and 0.01. Anything that
## reports a DMR count must read these same variables, which is why they are overridable:
## the number in the log and the dots on the page then cannot drift apart.
##
## 2026-10-03: raised to 50 and 0.01 on the user's call. At 10 percent the ideogram is a
## solid band on every chromosome and says nothing; 50 percent leaves the haplotype
## differences that are actually worth looking at.
methDiffCutoff <- as.numeric(Sys.getenv("IDEOGRAM_METHDIFF", "50"))
methDiffQValue <- as.numeric(Sys.getenv("IDEOGRAM_QVALUE",   "0.01"))
## Typography and canvas. The defaults reproduce the 12 pt, 8 x 7 in panel exactly; the
## Nature Methods render (paper_figures/fig_ed4) sets IDEOGRAM_BASE_PT=7 and a 3.5 in
## canvas, and blanks the title because the deck carries the panel caption.
basePt   <- as.numeric(Sys.getenv("IDEOGRAM_BASE_PT", "12"))
titlePt  <- as.numeric(Sys.getenv("IDEOGRAM_TITLE_PT", as.character(basePt + 2)))
figW     <- as.numeric(Sys.getenv("IDEOGRAM_WIDTH_IN", "8"))
figH     <- as.numeric(Sys.getenv("IDEOGRAM_HEIGHT_IN", "7"))
legendKey <- as.numeric(Sys.getenv("IDEOGRAM_LEGEND_KEY", "4"))
titleText <- Sys.getenv("IDEOGRAM_TITLE", "__default__")
cat(sprintf("[CUTOFF] methdiff >= %g, qvalue <= %g\n", methDiffCutoff, methDiffQValue))

## Parameterised copy of hpc_test/analysis/Figure4/IdeomgramPlot_methykit_{ont,pacbio}_hg002.R.
## The two originals differ only in platform name, input path, title and output name, so they
## are one script here. The originals are left untouched.
##
## Usage: 02_ideogram_dmr_blue.R <ont|pacbio> <outdir>
## commandArgs replaced for the notebook; everything else is the paper's file
argv <- c("pacbio", Sys.getenv("LV_OUT"))
platform <- if (length(argv) >= 1) argv[1] else "ont"
outdir   <- if (length(argv) >= 2) argv[2] else "."
stopifnot(platform %in% c("ont", "pacbio"))
dir.create(outdir, showWarnings = FALSE, recursive = TRUE)
PLATFORM_LABEL <- if (platform == "ont") "ONT" else "PacBio"

## The default is the June 2026 run. Its PacBio branch is primrose, so the PacBio
## ideogram must be pointed at a jasmine DMR directory instead; the ONT branch there
## is unaffected and stays the default. IDEOGRAM_WDIR overrides, nothing else changes.
wdir <- Sys.getenv('IDEOGRAM_WDIR',
                   '/project2/sli68423_1316/projects/long_verse/results/2026_06_16_phasing_dmr')
dir.create(wdir, showWarnings = FALSE, recursive = TRUE)
setwd(wdir)

chm13_chrom_size_file <- '/project2/sli68423_1316/users/yang/reference/chm13v2.0/chm13v2.0.chrom.sizes'
chm13_df <- read.table(chm13_chrom_size_file, sep = "\t", header = FALSE)

chm13.chr.len <- chm13_df[[2]]
names(chm13.chr.len) <- chm13_df[[1]]

# keep autosomes chr1-chr22 only
autosomes <- paste0("chr", 1:22)
chm13.chr.len <- chm13.chr.len[autosomes[autosomes %in% names(chm13.chr.len)]]

# load the methylKit DMR table. The originals name a plain .tsv; the files on disk were
# gzipped after that run, so take whichever is present rather than failing on the name.
dmr_file <- file.path(wdir, sprintf('methylkit_call/%s/hg002_%s_HP1_vs_HP2_methylkit_DMR_all.tsv',
                                    platform, platform))
if (!file.exists(dmr_file) && file.exists(paste0(dmr_file, '.gz'))) {
    dmr_file <- paste0(dmr_file, '.gz')
}
stopifnot(file.exists(dmr_file))
cat("DMR table:", dmr_file, "\n")
dmr_df <- read.table(gzfile(dmr_file), sep = "\t", header = TRUE,
                     stringsAsFactors = FALSE, check.names = FALSE)
dmr_df <- dmr_df[dmr_df$chr %in% names(chm13.chr.len),]

# gap regions (TSV with header: chrom, chromStart, chromEnd)
t2t_gap_bed_file <- file.path(wdir, 'AH107353_T2T_hgUnique.tsv')
t2t_gap_bed_df <- read.table(t2t_gap_bed_file, sep = "\t", header = TRUE, stringsAsFactors = FALSE)
t2t_gap_bed_df <- t2t_gap_bed_df[t2t_gap_bed_df$chrom %in% names(chm13.chr.len),]
t2t_gap_granges <- GRanges(
    seqnames = t2t_gap_bed_df$chrom,
    ranges = IRanges(start = t2t_gap_bed_df$chromStart, end = t2t_gap_bed_df$chromEnd)
)

filter_dmr <- function(dmr_df, difference, qvalue, type) {
    sig <- abs(dmr_df$meth.diff) >= difference & dmr_df$qvalue <= qvalue
    if (type == "hyper") {
        dmr_df[sig & dmr_df$meth.diff > 0,]
    } else {
        dmr_df[sig & dmr_df$meth.diff < 0,]
    }
}

dmr_to_granges <- function(dmr_df) {
    GRanges(
        seqnames = dmr_df$chr,
        ranges = IRanges(start = dmr_df$start, end = dmr_df$end),
        meth.diff = dmr_df$meth.diff
    )
}

gap_ylim <- c(-100, 100)

# colorblind-friendly palette (Wong): pink vs BLUE; gap stays red.
#
# The original used Wong yellow #F0E442 for hypo. Wong is colourblind safe, but yellow on
# white has almost no luminance contrast, and on a printed poster at viewing distance the
# hypo points disappear. Wong blue #0072B2 keeps the palette colourblind safe and is the
# darkest colour in it, so the two DMR classes separate by lightness as well as by hue,
# which is what survives printing and photocopying.
hyper_col <- "#CC79A7"  # pink
hypo_col <- "#0072B2"   # blue, was #F0E442 yellow
gap_col <- "red"

ideoDMC <- function(dmr_df, chrom.length, difference = 25,
                    qvalue = 0.01, circos = FALSE, title = "Test title",
                    hyper.col = hyper_col,
                    hypo.col = hypo_col,
                    gap.col = gap_col) {
    myIdeo <- GRanges(seqnames = names(chrom.length),
                      ranges = IRanges(start = 1, width = chrom.length))
    seqlevels(myIdeo) <- names(chrom.length)
    seqlengths(myIdeo) <- chrom.length

    hyper <- filter_dmr(dmr_df, difference, qvalue, "hyper")
    hypo <- filter_dmr(dmr_df, difference, qvalue, "hypo")

    g.per <- dmr_to_granges(hyper)
    seqlevels(g.per) <- seqlevels(myIdeo)
    seqlengths(g.per) <- chrom.length

    g.po <- dmr_to_granges(hypo)
    seqlevels(g.po) <- seqlevels(myIdeo)
    seqlengths(g.po) <- chrom.length

    values(g.po)$id <- "Hypo"
    values(g.per)$id <- "Hyper"
    legend_levels <- c("Hyper", "Hypo", "Gap region")
    values(g.po)$id <- factor(values(g.po)$id, levels = legend_levels)
    values(g.per)$id <- factor(values(g.per)$id, levels = legend_levels)

    # invisible dummy point so "Gap region" appears in the color legend
    g.gap.legend <- GRanges(
        seqnames = names(chrom.length)[1],
        ranges = IRanges(start = 1, width = 1),
        meth.diff = gap_ylim[1] - 1,
        id = factor("Gap region", levels = legend_levels)
    )
    seqlevels(g.gap.legend) <- seqlevels(myIdeo)
    seqlengths(g.gap.legend) <- chrom.length
    g.dmr <- c(g.po, g.per, g.gap.legend)

    if (circos) {
        p <- ggplot() + layout_circle(myIdeo, geom = "ideo", fill = "gray70",
                                      radius = 39, trackWidth = 2)

        p <- p +
            layout_circle(c(g.po, g.per), geom = "point",
                          size = 1, aes(x = midpoint,
                                        y = meth.diff, color = id), radius = 25, trackWidth = 30) +
            scale_colour_manual(values = c("Hyper" = hyper.col, "Hypo" = hypo.col))
        p3 <- p +
            layout_circle(myIdeo, geom = "text", aes(label = seqnames),
                          vjust = 0, radius = 55, trackWidth = 7) +
            labs(title = title)

    } else {
        legend_shapes <- c("Hyper" = 16, "Hypo" = 16, "Gap region" = 22)
        legend_title <- "Annotation"
        p1 <- ggplot() + layout_karyogram(myIdeo)
        p2 <- p1 +
            layout_karyogram(g.dmr, geom = "point", size = 0.01,
                             aes(x = midpoint,
                                 y = meth.diff, color = id)) +
            scale_colour_manual(legend_title,
                                breaks = legend_levels,
                                values = c("Hyper" = hyper.col,
                                           "Hypo" = hypo.col,
                                           "Gap region" = gap.col)) +
            labs(title = title) +
            xlab("Genomic coordinates") +
            ylab("Methylation difference (%)") +
            ## ggplot's base_size is not a floor: axis.text defaults to rel(0.8), so a
            ## base of 11 puts the chromosome strip labels and both axes at 8.8 pt. Only
            ## plot.title was ever set here, which is why the panel sat at 9 pt while
            ## reading as finished. Every text element is named explicitly instead.
            theme(
                plot.title   = if (nzchar(title)) element_text(hjust = 0.5, size = titlePt) else element_blank(),
                axis.title   = element_text(size = basePt),
                axis.text    = element_text(size = basePt),
                strip.text   = element_text(size = basePt),
                legend.title = element_text(size = basePt),
                legend.text  = element_text(size = basePt),
                legend.key.size = unit(basePt * 1.2, "pt")
            ) +
            guides(color = guide_legend(
                title = legend_title,
                override.aes = list(
                    shape = unname(legend_shapes[legend_levels]),
                    fill = c(hyper.col, hypo.col, NA),
                    colour = c(hyper.col, hypo.col, gap.col),
                    alpha = c(1, 1, 1),
                    size = c(legendKey, legendKey, legendKey)
                )
            ))
        p3 <- p2 +
            layout_karyogram(t2t_gap_granges,
                             geom = "rect",
                             rect.height = abs(gap_ylim[2] - gap_ylim[1]) / 2,
                             ylim = gap_ylim,
                             color = gap.col, fill = NA) +
            coord_cartesian(ylim = gap_ylim)
    }

    return(p3)
}

ideoPlot <- ideoDMC(dmr_df, chrom.length = chm13.chr.len,
                    difference = methDiffCutoff, qvalue = methDiffQValue,
                    circos = FALSE,
                    title = if (identical(titleText, "__default__"))
                        sprintf("%s: DMRs of Haplotypes in HG002", PLATFORM_LABEL) else titleText)

ideoPlot

stem <- file.path(outdir, sprintf('Ideogram_dmr_t2t_hg002_%s_blue', platform))

## cairo_pdf, not pdf(): the base device does not embed fonts, so an Arial panel opens in
## whatever the reader substitutes. Same reason png() is given the cairo type. The size is
## 8 x 7 rather than 8 x 6: at 12 pt the 24 chromosome strip labels need the extra inch.
cairo_pdf(paste0(stem, '.pdf'), width = figW, height = figH)
print(ideoPlot)
dev.off()

png(paste0(stem, '.png'), width = figW, height = figH, units = "in", res = 400, type = "cairo")
print(ideoPlot)
dev.off()

cat("wrote", paste0(stem, c('.pdf', '.png')), "\n")
cat("hyper", hyper_col, " hypo", hypo_col, "\n")

## f, g - imprinting control regions, HP1 minus HP2

ComplexHeatmap. The script checks its own output row by row against a reference table before it writes, so a silent change in the numbers stops it rather than producing a plausible figure.

In [ ]:
%%R
## scripts/paper/icr_heatmap_nm.R, run as R. Edit any line and re-run the cell.
Sys.setenv(ICR_BASE_PT = "7")

#!/usr/bin/env Rscript
## Imprinting-control-region heatmaps (HP1 minus HP2 methylation, ONT and PacBio) at Nature
## Methods size: 7 pt text, cells sized so the 43-column novel heatmap spans the 183 mm page.
##
## Parameterised copy of results/2026_10_03_2platform_phasing_jasmine/
## _2025_06_17_figure5_ab_plot_heatmap_compare_icr.R (the 12 pt deck version). Same input,
## same ordering (ONT difference descending), same colour scale; the plot tables it writes
## are checked row for row against that script's plotdf.csv, so the two renders cannot
## drift apart in content.
##
## Usage: 12_icr_heatmap_nm.R <input merged csv> <reference dir with *.plotdf.csv> <outdir>
## Env:   ICR_BASE_PT (7), ICR_CELL_W_MM (3.3), ICR_CELL_H_MM (4), ICR_COL_LABEL_MM (22)
suppressPackageStartupMessages({
    library(tidyverse); library(ComplexHeatmap); library(circlize); library(RColorBrewer); library(grid)
})
## commandArgs replaced for the notebook; everything else is the paper's file
argv <- c(file.path(Sys.getenv("LV_DATA"), "ed_fig4/hg002_ONT_Pacbio_merged_with_annotation.mincov5.inner.csv"), file.path(Sys.getenv("LV_DATA"), "ed_fig4"), Sys.getenv("LV_OUT"))
stopifnot(length(argv) == 3)
input_file <- argv[1]; ref_dir <- argv[2]; outdir <- argv[3]
dir.create(outdir, showWarnings = FALSE, recursive = TRUE)

BASE_PT <- as.numeric(Sys.getenv("ICR_BASE_PT", "7"))
CELL_W_MM <- as.numeric(Sys.getenv("ICR_CELL_W_MM", "3.3"))
CELL_H_MM <- as.numeric(Sys.getenv("ICR_CELL_H_MM", "4"))
ROW_GAP_MM <- 1
LEGEND_W_MM <- 16
ROW_LABEL_W_MM <- 11
COL_LABEL_H_MM <- as.numeric(Sys.getenv("ICR_COL_LABEL_MM", "22"))
MARGIN_MM <- 2

df_all <- read_csv(input_file, show_col_types = FALSE) %>%
    transmute(Name = as.character(Name), Status = as.character(Status),
              ONT_diff = as.numeric(ONT_HP_diff), Pacbio_diff = as.numeric(Pacbio_HP_diff)) %>%
    drop_na(ONT_diff, Pacbio_diff) %>%
    arrange(desc(ONT_diff), desc(Pacbio_diff))
cat("rows:", nrow(df_all), " known:", sum(df_all$Status == "Known"), " novel:", sum(df_all$Status == "Novel"), "\n")

breaks <- seq(0, 1, length.out = 11)
col_fun <- colorRamp2(breaks, colorRampPalette(rev(brewer.pal(n = 7, name = "RdYlBu")))(length(breaks)))

plotdf_known <- df_all %>% filter(Status == "Known") %>% select(Name, ONT = ONT_diff, PacBio = Pacbio_diff)
plotdf_novel <- df_all %>% filter(Status == "Novel") %>% select(Name, ONT = ONT_diff, PacBio = Pacbio_diff) %>%
    mutate(Name = str_replace(Name, "intragenic_", "intra_"),
           Name = str_replace(Name, "PWARSN,SNORD107,PWARSN,PWAR5", "PWARSN,SNORD107"))

check_against_reference <- function(plotdf, ref_csv) {
    ref <- read_csv(ref_csv, show_col_types = FALSE)
    stopifnot(nrow(ref) == nrow(plotdf), all(ref$Name == plotdf$Name),
              max(abs(ref$ONT - plotdf$ONT)) < 1e-9, max(abs(ref$PacBio - plotdf$PacBio)) < 1e-9)
    cat("  matches", basename(ref_csv), "(", nrow(ref), "regions )\n")
}
check_against_reference(plotdf_known, file.path(ref_dir, "hg002_known_icr_hp_diff_heatmap.mincov5.plotdf.csv"))
check_against_reference(plotdf_novel, file.path(ref_dir, "hg002_novel_icr_hp_diff_heatmap.mincov5.plotdf.csv"))

draw_one <- function(plotdf, tag) {
    write_csv(plotdf, file.path(outdir, sprintf("icr_heatmap_%s_nm.plotdf.csv", tag)))
    mat <- plotdf %>% column_to_rownames("Name") %>% as.matrix() %>% t()
    n_col <- ncol(mat); n_row <- nrow(mat)
    italic_labels <- parse(text = paste0("italic('", plotdf$Name, "')"))
    ht <- Heatmap(mat, name = "HP1 - HP2", col = col_fun, na_col = "black",
                  width = unit(n_col * CELL_W_MM, "mm"),
                  height = unit(n_row * CELL_H_MM + (n_row - 1L) * ROW_GAP_MM, "mm"),
                  column_labels = italic_labels, column_title = character(0),
                  cluster_columns = FALSE, cluster_rows = FALSE, row_title = NULL,
                  column_names_gp = gpar(fontsize = BASE_PT), row_names_gp = gpar(fontsize = BASE_PT),
                  heatmap_legend_param = list(title_gp = gpar(fontsize = BASE_PT, fontface = "bold"),
                                              labels_gp = gpar(fontsize = BASE_PT),
                                              grid_width = unit(3, "mm"), legend_height = unit(14, "mm")),
                  row_split = factor(c("ONT", "PacBio"), levels = c("ONT", "PacBio")),
                  row_gap = unit(ROW_GAP_MM, "mm"), rect_gp = gpar(col = "#DDDDDD", lwd = 0.5))
    w_in <- (n_col * CELL_W_MM + LEGEND_W_MM + ROW_LABEL_W_MM + 2 * MARGIN_MM) / 25.4
    ## the legend is centred on the two heatmap rows and its title rises above them: 6 mm of
    ## headroom, or the title is clipped at the top of the device (seen 2026-10-03)
    TOP_MM <- MARGIN_MM + 6
    h_in <- (n_row * CELL_H_MM + (n_row - 1L) * ROW_GAP_MM + COL_LABEL_H_MM + MARGIN_MM + TOP_MM) / 25.4
    stem <- file.path(outdir, sprintf("icr_heatmap_%s_nm", tag))
    pad <- unit(c(MARGIN_MM, MARGIN_MM, TOP_MM, MARGIN_MM), "mm")   # bottom, left, top, right
    cairo_pdf(paste0(stem, ".pdf"), width = w_in, height = h_in); draw(ht, padding = pad); dev.off()
    png(paste0(stem, ".png"), width = w_in, height = h_in, units = "in", res = 300, type = "cairo")
    draw(ht, padding = pad); dev.off()
    cat(sprintf("wrote %s.pdf/.png  %d regions, %.2f x %.2f in, %g pt\n", stem, n_col, w_in, h_in, BASE_PT))
}
draw_one(plotdf_novel, "novel")
draw_one(plotdf_known, "known")